# BÀI TẬP — Transfer Learning với EfficientNet-B0 trên FGVC-Aircraft

**Yêu cầu:** huấn luyện mô hình cho bộ dữ liệu **FGVC-Aircraft** bằng **EfficientNet-B0 pretrained ImageNet-1K**.

Quy trình giống Bài 6: Linear probe → Partial fine-tuning → Full fine-tuning, **chọn mô hình bằng validation loss**, **đánh giá test đúng một lần**, phân tích lỗi, lưu/tải checkpoint.

**Môi trường:** Google Colab, Runtime → Change runtime type → **T4 GPU**, rồi chạy lần lượt từng ô.

**Dữ liệu:** `torchvision.datasets.FGVCAircraft`, mức nhãn `variant` (100 biến thể máy bay), chia chính thức train 3.334 / val 3.333 / test 3.333 ảnh (khoảng 2,7 GB, lần đầu tải mất vài phút).

## 0. Thiết lập và tái lập

In [ ]:
import copy
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models, transforms
from torchvision.models import EfficientNet_B0_Weights

SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 2
IMAGE_SIZE = 224
ANNOTATION_LEVEL = "variant"      # "variant" (100 lớp) | "family" (70) | "manufacturer" (30)
EPOCHS_PROBE = 30
EPOCHS_PARTIAL = 10
EPOCHS_FULL = 10
PATIENCE = 4
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./baitap_aircraft_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Cần GPU: Runtime → Change runtime type → T4 GPU."
DEVICE = torch.device("cuda:0")
WEIGHTS = EfficientNet_B0_Weights.IMAGENET1K_V1


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
print("Torch:", torch.__version__, "| Thiết bị:", torch.cuda.get_device_name(0))

## 1. Dữ liệu FGVC-Aircraft

Mỗi ảnh FGVC-Aircraft có **dải bản quyền cao 20 pixel ở đáy ảnh**, nên ta cắt bỏ trước khi tiền xử lý để mô hình không học từ phần chữ này.

- **Train:** cắt dải → `RandomResizedCrop` → lật ngang → ColorJitter nhẹ → Normalize ImageNet.
- **Val/Test:** cắt dải → `weights.transforms()` (Resize 256 → CenterCrop 224 → Normalize), xác định (deterministic).

In [ ]:
def crop_banner(img):
    """Cắt dải copyright 20 px ở đáy ảnh FGVC-Aircraft."""
    w, h = img.size
    return img.crop((0, 0, w, h - 20))


MEAN, STD = WEIGHTS.transforms().mean, WEIGHTS.transforms().std

train_transform = transforms.Compose([
    transforms.Lambda(crop_banner),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Lambda(crop_banner),
    WEIGHTS.transforms(),
])

kw = dict(root=DATA_DIR, annotation_level=ANNOTATION_LEVEL)
aircraft_train = datasets.FGVCAircraft(split="train", transform=train_transform, download=True, **kw)
aircraft_train_clean = datasets.FGVCAircraft(split="train", transform=eval_transform, download=False, **kw)
aircraft_val = datasets.FGVCAircraft(split="val", transform=eval_transform, download=False, **kw)

CLASS_NAMES = aircraft_train.classes
NUM_CLASSES = len(CLASS_NAMES)
print("Train:", len(aircraft_train), "| Val:", len(aircraft_val), "| Số lớp:", NUM_CLASSES)
print("Một vài lớp:", CLASS_NAMES[:5])

train_loader = DataLoader(aircraft_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
train_clean_loader = DataLoader(aircraft_train_clean, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(aircraft_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

images, labels = next(iter(train_loader))
print("Một batch:", tuple(images.shape), "| Nhãn:", tuple(labels.shape))
assert images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)

In [ ]:
def show_image(ax, tensor, title):
    arr = tensor.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)
    ax.imshow(np.clip(arr, 0, 1)); ax.set_title(title, fontsize=8); ax.axis("off")


fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, idx in zip(axes.flat, np.linspace(0, len(aircraft_train_clean) - 1, 10, dtype=int)):
    img, target = aircraft_train_clean[int(idx)]
    show_image(ax, img, CLASS_NAMES[target])
fig.suptitle("FGVC-Aircraft: ví dụ từ tập train")
plt.tight_layout(); plt.show()

## 2. EfficientNet-B0: backbone, head và số tham số

`torchvision` EfficientNet-B0 gồm `features` (stem + 7 khối MBConv + conv cuối, đánh số `features[0]…features[8]`), `avgpool` và `classifier = Sequential(Dropout, Linear(1280, 1000))`. Ta thay `classifier[1]` bằng `Linear(1280, 100)`.

Ba chiến lược:
- **probe:** chỉ học `classifier`;
- **partial:** mở `features[6:]` (khối MBConv cuối + conv 1×1 cuối) và `classifier`;
- **full:** mở toàn mạng với learning rate theo nhóm tầng.

In [ ]:
def count_parameters(model: nn.Module) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"Total": total, "Trainable": trainable, "Frozen": total - trainable}


def build_efficientnet(num_classes: int = NUM_CLASSES, pretrained: bool = True) -> nn.Module:
    model = models.efficientnet_b0(weights=WEIGHTS if pretrained else None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model


def configure_trainable(model: nn.Module, strategy: str) -> nn.Module:
    if strategy not in ("probe", "partial", "full"):
        raise ValueError(strategy)
    for p in model.parameters():
        p.requires_grad = (strategy == "full")
    for p in model.classifier.parameters():
        p.requires_grad = True
    if strategy == "partial":
        for p in model.features[6:].parameters():
            p.requires_grad = True
    return model


def set_training_mode(model: nn.Module, strategy: str) -> None:
    """train() cho toàn mạng, nhưng BatchNorm của các tầng đóng băng giữ ở eval()."""
    model.train()
    if strategy in ("probe", "partial"):
        for m in model.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                if not any(p.requires_grad for p in m.parameters()):
                    m.eval()


def build_optimizer(model: nn.Module, strategy: str) -> torch.optim.Optimizer:
    if strategy == "probe_head":     # head tuyến tính rời, học trên embeddings đã cache
        return torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-3)
    if strategy == "probe":
        return torch.optim.AdamW(model.classifier.parameters(), lr=3e-3, weight_decay=1e-3)
    if strategy == "partial":
        return torch.optim.AdamW([
            {"params": model.features[6:].parameters(), "lr": 1e-4},
            {"params": model.classifier.parameters(), "lr": 1e-3},
        ], weight_decay=1e-4)
    if strategy == "full":
        return torch.optim.AdamW([
            {"params": model.features[:3].parameters(), "lr": 1e-5},
            {"params": model.features[3:6].parameters(), "lr": 3e-5},
            {"params": model.features[6:].parameters(), "lr": 1e-4},
            {"params": model.classifier.parameters(), "lr": 1e-3},
        ], weight_decay=1e-4)
    raise ValueError(strategy)


demo = build_efficientnet(pretrained=False)
rows = []
for name, st in [("Linear probe (classifier)", "probe"), ("Partial (features[6:] + classifier)", "partial"),
                 ("Full (toàn mạng)", "full")]:
    rows.append({"Chiến lược": name, **count_parameters(configure_trainable(copy.deepcopy(demo), st))})
display(pd.DataFrame(rows))
assert demo.classifier[1].out_features == NUM_CLASSES
assert rows[0]["Trainable"] < rows[1]["Trainable"] < rows[2]["Trainable"]
del demo

## 3. Hàm huấn luyện / đánh giá

- Loss là `CrossEntropyLoss` trên logits, tổng hợp theo số mẫu.
- Dùng mixed precision (`torch.autocast` + `GradScaler`) cho nhanh trên T4.
- Chọn checkpoint tốt nhất theo **validation loss**, có early stopping.

In [ ]:
def evaluate(model, loader, num_classes=NUM_CLASSES, use_amp=True):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    total_loss, total, c1, c5 = 0.0, 0, 0, 0
    all_t, all_p = [], []
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            logits = logits.float()
            total_loss += criterion(logits, y).item()
            pred = logits.argmax(1)
            c1 += int((pred == y).sum())
            c5 += int((logits.topk(5, dim=1).indices == y.unsqueeze(1)).any(1).sum())
            total += y.size(0)
            all_t.extend(y.cpu().tolist()); all_p.extend(pred.cpu().tolist())
    return {"loss": total_loss / total, "accuracy": c1 / total, "top5": c5 / total,
            "macro_f1": f1_score(all_t, all_p, labels=list(range(num_classes)),
                                 average="macro", zero_division=0),
            "targets": np.asarray(all_t), "preds": np.asarray(all_p)}


def fit_model(name, model, train_data, val_data, strategy, epochs, use_amp=True):
    model = model.to(DEVICE)
    optimizer = build_optimizer(model, strategy)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    criterion = nn.CrossEntropyLoss()
    history, best_state = [], None
    best_loss, best_epoch, patience_count = float("inf"), 0, 0
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        set_training_mode(model, strategy)
        total_loss, seen = 0.0, 0
        for x, y in train_data:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            loss = criterion(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.item()) * len(y)
            seen += len(y)
        vm = evaluate(model, val_data, use_amp=use_amp)
        rec = {"epoch": epoch, "train_loss": total_loss / seen, "val_loss": vm["loss"],
               "val_acc": vm["accuracy"], "val_f1": vm["macro_f1"], "val_top5": vm["top5"]}
        history.append(rec)
        print(f"[{name}] {epoch:02d}/{epochs:02d} | train_loss={rec['train_loss']:.4f} | "
              f"val_loss={rec['val_loss']:.4f} | val_acc={rec['val_acc']:.2%} | val_F1={rec['val_f1']:.4f}")
        if vm["loss"] < best_loss - 1e-6:
            best_loss, best_epoch, patience_count = vm["loss"], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
        scheduler.step()
        if patience_count >= PATIENCE:
            print(f"[{name}] Early stopping tại epoch {epoch}.")
            break
    model.load_state_dict(best_state)
    m = evaluate(model, val_data, use_amp=use_amp)
    summary = {"Mô hình": name, "Best epoch": best_epoch, "Val loss": m["loss"],
               "Val accuracy": m["accuracy"], "Val macro-F1": m["macro_f1"],
               "Val top-5": m["top5"], "Thời gian (s)": round(time.perf_counter() - start, 1),
               "Trainable": count_parameters(model)["Trainable"]}
    print(f"[{name}] BEST epoch={best_epoch}, val_loss={m['loss']:.4f}, val_acc={m['accuracy']:.2%}")
    return model, pd.DataFrame(history), summary


def plot_histories(histories, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for name, f in histories.items():
        axes[0].plot(f.epoch, f.val_acc, marker="o", label=name)
        axes[1].plot(f.epoch, f.train_loss, "--", alpha=0.5, label=f"{name} train")
        axes[1].plot(f.epoch, f.val_loss, marker="o", label=f"{name} val")
    axes[0].set(xlabel="Epoch", ylabel="Accuracy", title="Validation accuracy", ylim=(0, 1))
    axes[1].set(xlabel="Epoch", ylabel="Cross-entropy", title="Train loss / Val loss")
    for ax in axes:
        ax.grid(alpha=0.2); ax.legend(fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

## 4. Thí nghiệm B — Frozen EfficientNet-B0 + Linear probing

Backbone đóng băng + biến đổi deterministic ⇒ cache embedding **1280 chiều** một lần, sau đó chỉ train một lớp tuyến tính.

In [ ]:
def extract_features(pretrained_model, loader):
    backbone = copy.deepcopy(pretrained_model)
    backbone.classifier = nn.Identity()          # đầu ra: vector 1280 chiều sau avgpool + flatten
    backbone = backbone.to(DEVICE).eval()
    vecs, labs = [], []
    with torch.inference_mode():
        for x, y in loader:
            vecs.append(backbone(x.to(DEVICE, non_blocking=True)).float().cpu())
            labs.append(y.clone())
    del backbone
    return TensorDataset(torch.cat(vecs), torch.cat(labs))


class LinearProbe(nn.Module):
    def __init__(self, in_features=1280, out_features=NUM_CLASSES):
        super().__init__()
        self.classifier = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.classifier(x)


histories, fitted_models, summaries = {}, {}, {}

pretrained_probe = build_efficientnet(pretrained=True)
cached_train = extract_features(pretrained_probe, train_clean_loader)
cached_val = extract_features(pretrained_probe, val_loader)
print("Cached train:", tuple(cached_train.tensors[0].shape), "| Cached val:", tuple(cached_val.tensors[0].shape))
probe_train_loader = DataLoader(cached_train, batch_size=128, shuffle=True)
probe_val_loader = DataLoader(cached_val, batch_size=256, shuffle=False)

set_seed(SEED)
probe_head = LinearProbe()
# fit_model dùng evaluate với autocast; head rời không cần AMP -> use_amp=False
probe_head, histories["Linear probe"], summaries["Linear probe"] = fit_model(
    "Linear probe", probe_head, probe_train_loader, probe_val_loader,
    strategy="probe_head", epochs=EPOCHS_PROBE, use_amp=False)

# Ghép head đã học vào backbone pretrained để suy luận trên ảnh
pretrained_probe.classifier[1].load_state_dict(probe_head.classifier.cpu().state_dict())
fitted_models["Linear probe"] = configure_trainable(pretrained_probe.cpu(), "probe")
summaries["Linear probe"]["Trainable"] = count_parameters(fitted_models["Linear probe"])["Trainable"]
display(pd.DataFrame([summaries["Linear probe"]]).round(4))
del probe_head, cached_train, cached_val
torch.cuda.empty_cache()

## 5. Thí nghiệm C — Partial fine-tuning (`features[6:]` + classifier)

In [ ]:
set_seed(SEED)
partial_model = configure_trainable(build_efficientnet(pretrained=True), "partial")
print("Số tham số partial:", count_parameters(partial_model))
partial_model, histories["Partial FT"], summaries["Partial FT"] = fit_model(
    "Partial FT", partial_model, train_loader, val_loader, strategy="partial", epochs=EPOCHS_PARTIAL)
fitted_models["Partial FT"] = partial_model.cpu()
display(pd.DataFrame([summaries["Partial FT"]]).round(4))
del partial_model
torch.cuda.empty_cache()

## 6. Thí nghiệm D — Full fine-tuning với learning rate theo tầng

LR: `features[0:3]` = 1e-5, `features[3:6]` = 3e-5, `features[6:]` = 1e-4, `classifier` = 1e-3.

In [ ]:
set_seed(SEED)
full_model = configure_trainable(build_efficientnet(pretrained=True), "full")
print("Learning rates:", [g["lr"] for g in build_optimizer(full_model, "full").param_groups])
full_model, histories["Full FT"], summaries["Full FT"] = fit_model(
    "Full FT", full_model, train_loader, val_loader, strategy="full", epochs=EPOCHS_FULL)
fitted_models["Full FT"] = full_model.cpu()
display(pd.DataFrame([summaries["Full FT"]]).round(4))
del full_model
torch.cuda.empty_cache()

## 7. So sánh trên validation và chọn mô hình

Xếp theo `Val loss` tăng dần (hòa thì theo `Val macro-F1` giảm dần). Chỉ mô hình đứng đầu bảng được đưa sang test.

In [ ]:
comparison = pd.DataFrame(summaries.values()).sort_values(
    ["Val loss", "Val macro-F1"], ascending=[True, False]).reset_index(drop=True)
display(comparison.round(4))
selected_name = str(comparison.iloc[0]["Mô hình"])
best_model = fitted_models[selected_name].to(DEVICE).eval()
print("Mô hình được chọn CHỈ từ validation:", selected_name)
plot_histories(histories, "EfficientNet-B0 trên FGVC-Aircraft: probe / partial / full")
comparison.to_csv(OUTPUT_DIR / "validation_comparison.csv", index=False)

## 8. Đánh giá test đúng một lần

In [ ]:
aircraft_test = datasets.FGVCAircraft(split="test", transform=eval_transform, download=False, **kw)
test_loader = DataLoader(aircraft_test, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
test_metrics = evaluate(best_model, test_loader)
test_report = pd.DataFrame([{
    "Selected model": selected_name,
    "Test loss": test_metrics["loss"],
    "Test top-1 accuracy": test_metrics["accuracy"],
    "Test macro-F1": test_metrics["macro_f1"],
    "Test top-5 accuracy": test_metrics["top5"],
    "N test": len(aircraft_test),
}])
display(test_report.round(4))
test_report.to_csv(OUTPUT_DIR / "final_test_metrics.csv", index=False)

## 9. Phân tích sai số: confusion matrix và lớp khó

In [ ]:
cm = confusion_matrix(test_metrics["targets"], test_metrics["preds"], labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set(xlabel="Lớp dự đoán", ylabel="Lớp thật", title=f"Confusion matrix — {selected_name} ({NUM_CLASSES} lớp)")
plt.tight_layout(); plt.show()

report = classification_report(test_metrics["targets"], test_metrics["preds"],
                               labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES,
                               zero_division=0, output_dict=True)
class_table = pd.DataFrame(report).T.iloc[:NUM_CLASSES]
print("10 lớp có F1 thấp nhất:")
display(class_table.sort_values("f1-score").head(10).round(3))
class_table.to_csv(OUTPUT_DIR / "test_per_class_metrics.csv")

# 10 cặp nhầm lẫn nhiều nhất (thật -> dự đoán)
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
top_pairs = np.dstack(np.unravel_index(np.argsort(-cm_off.ravel())[:10], cm_off.shape))[0]
print("10 cặp nhầm lẫn nhiều nhất:")
display(pd.DataFrame([{"Thật": CLASS_NAMES[t], "Dự đoán": CLASS_NAMES[p], "Số ảnh": int(cm_off[t, p])}
                      for t, p in top_pairs]))

In [ ]:
mistakes = np.where(test_metrics["targets"] != test_metrics["preds"])[0]
print("Tổng ảnh dự đoán sai:", len(mistakes), "/", len(aircraft_test),
      f"({len(mistakes) / len(aircraft_test):.2%})")
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, idx in zip(axes.flat, mistakes[:8]):
    tensor, true_label = aircraft_test[int(idx)]
    show_image(ax, tensor, f"Thật: {CLASS_NAMES[true_label]}\nDự đoán: {CLASS_NAMES[test_metrics['preds'][idx]]}")
plt.tight_layout(); plt.show()

## 10. Lưu / tải checkpoint và suy luận một ảnh

In [ ]:
checkpoint_path = OUTPUT_DIR / "aircraft_best_efficientnet_b0.pt"
torch.save({
    "architecture": "efficientnet_b0", "num_classes": NUM_CLASSES,
    "annotation_level": ANNOTATION_LEVEL, "class_names": CLASS_NAMES,
    "selected_strategy": selected_name,
    "state_dict": {k: v.detach().cpu() for k, v in best_model.state_dict().items()},
    "image_size": IMAGE_SIZE, "normalization_mean": list(MEAN), "normalization_std": list(STD),
    "val_loss": float(comparison.iloc[0]["Val loss"]),
}, checkpoint_path)

payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored = build_efficientnet(num_classes=payload["num_classes"], pretrained=False)
restored.load_state_dict(payload["state_dict"])
restored = restored.to(DEVICE).eval()

x, y = aircraft_test[0]
with torch.inference_mode():
    before = best_model(x.unsqueeze(0).to(DEVICE)).cpu()
    after = restored(x.unsqueeze(0).to(DEVICE)).cpu()
assert torch.allclose(before, after, atol=1e-4, rtol=1e-4)
probs = after.softmax(1)[0]
vals, idxs = probs.topk(5)
display(pd.DataFrame({"Lớp dự đoán": [payload["class_names"][i] for i in idxs.tolist()],
                      "Xác suất": vals.tolist()}))
print("Nhãn thật:", payload["class_names"][y])
print("Checkpoint:", checkpoint_path, "| MB:", round(checkpoint_path.stat().st_size / 1024**2, 1))
print("Reload khớp logits:", bool(torch.allclose(before, after, atol=1e-4, rtol=1e-4)))

## 11. Nhận xét (điền sau khi chạy)

1. So sánh ba chiến lược trên validation (loss, accuracy, macro-F1, top-5). Chiến lược nào được chọn và vì sao?
2. FGVC-Aircraft là bài toán **phân loại chi tiết** (các biến thể máy bay rất giống nhau). Vì sao fine-tuning thường cần nhiều hơn linear probe ở bộ dữ liệu này?
3. Những cặp lớp nào bị nhầm lẫn nhiều nhất? Giả thuyết nguyên nhân (cần kiểm chứng, không khẳng định bừa).
4. Giới hạn: một seed, số epoch khác nhau, chưa tuning siêu tham số.
5. Mở rộng: thử `annotation_level="family"` hoặc `"manufacturer"`, thử `trainval`, gradual unfreezing, hoặc backbone khác (EfficientNet-B3, ConvNeXt).